# 128. VAE（変分オートエンコーダ）をゼロから実装する

生成モデルの入門として、**VAE（Variational Autoencoder）** を PyTorch でゼロから実装します。

## このノートブックで学ぶこと

1. オートエンコーダ（AE）と VAE の違い
2. 変分推論と ELBO（証拠下界）の理論
3. 再パラメータ化トリック（Reparameterization Trick）
4. Fashion-MNIST での VAE 学習と評価
5. 潜在空間の可視化と画像生成

## 対象者

- PyTorch の基礎（テンソル、nn.Module）を理解している方
- オートエンコーダの仕組みを知っている方
- 生成モデルに興味がある方

---

**参考文献**
- Kingma & Welling (2013): *Auto-Encoding Variational Bayes* https://arxiv.org/abs/1312.6114
- Doersch (2016): *Tutorial on Variational Autoencoders* https://arxiv.org/abs/1606.05908

## 1. 理論背景

### 1.1 オートエンコーダ（AE）vs VAE

| | オートエンコーダ（AE） | VAE |
|---|---|---|
| 潜在変数 z | 決定論的なベクトル | 確率分布 q(z|x) からのサンプル |
| エンコーダ出力 | z （点推定） | μ, σ² （分布のパラメータ） |
| 学習目的 | 再構成誤差の最小化 | ELBO の最大化 |
| 生成能力 | 限定的（潜在空間に穴がある） | 滑らかな潜在空間で高い生成能力 |

普通のオートエンコーダは潜在空間が不連続になりがちで、任意の点 z からサンプリングしても意味のある画像が生成できません。
VAE は潜在空間に正則化をかけることで、**滑らかで連続的な潜在空間** を学習します。

---

### 1.2 潜在空間 z の確率的解釈

VAE は以下の生成モデルを仮定します：

$$p(x) = \int p(x|z) \, p(z) \, dz$$

- **事前分布** $p(z) = \mathcal{N}(0, I)$：潜在変数は標準正規分布に従う
- **尤度** $p(x|z)$：デコーダが表現する条件付き分布
- **事後分布** $p(z|x)$：真の事後分布（通常計算不可能）

真の事後分布 $p(z|x)$ は計算不可能なため、近似分布 $q(z|x) = \mathcal{N}(\mu, \sigma^2 I)$ で近似します（**変分推論**）。

---

### 1.3 ELBO の導出

対数尤度 $\log p(x)$ を最大化したいですが、直接計算できません。
KL ダイバージェンスの非負性から下界（ELBO）を導出します：

$$\log p(x) = \mathcal{L}(\theta, \phi; x) + D_{KL}[q_\phi(z|x) \| p_\theta(z|x)]$$

$$\log p(x) \geq \mathcal{L}(\theta, \phi; x) = \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - D_{KL}[q_\phi(z|x) \| p(z)]$$

この **ELBO（Evidence Lower BOund）** を最大化します：

$$\mathcal{L} = \underbrace{\mathbb{E}[\log p(x|z)]}_{\text{再構成誤差}} - \underbrace{D_{KL}[q(z|x) \| p(z)]}_{\text{KL 正則化}}$$

- **再構成誤差**：入力 x をどれだけ正確に復元できるか
- **KL 正則化**：近似事後分布を事前分布 N(0,I) に近づける

正規分布同士の KL は解析的に計算できます：

$$D_{KL}[\mathcal{N}(\mu, \sigma^2) \| \mathcal{N}(0, I)] = -\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

---

### 1.4 再パラメータ化トリック

**問題**: z のサンプリング操作 $z \sim q(z|x)$ は微分不可能なため、誤差逆伝播できません。

**解決策**: ランダム性を別の変数 ε に分離します：

$$z = \mu + \epsilon \cdot \sigma, \quad \epsilon \sim \mathcal{N}(0, I)$$

これにより、勾配を μ と σ に対して計算できます。

```
入力 x
  ↓
エンコーダ  →  μ, log σ²
                ↓          ↑ε ~ N(0,I)
         z = μ + ε·exp(log σ²/2)
                ↓
デコーダ  →  x̂（再構成）
```

---

### 1.5 β-VAE

β-VAE は KL 損失に重み β をかけることで潜在空間の解釈性を向上させます：

$$\mathcal{L}_{\beta} = \mathbb{E}[\log p(x|z)] - \beta \cdot D_{KL}[q(z|x) \| p(z)]$$

- β > 1：より解釈可能な潜在表現（ただし再構成品質は低下）
- β = 1：標準的な VAE

In [ ]:
# セットアップ: 必要なライブラリのインポート
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

# 再現性のためのシード設定
torch.manual_seed(42)
np.random.seed(42)

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

In [ ]:
# VAE モデル定義

class VAE(nn.Module):
    """変分オートエンコーダ（Variational Autoencoder）
    
    Args:
        input_dim  (int): 入力次元数（デフォルト: 784 = 28x28）
        hidden_dim (int): 隠れ層の次元数（デフォルト: 512）
        latent_dim (int): 潜在空間の次元数（デフォルト: 2）
    """
    
    def __init__(self, input_dim: int = 784, hidden_dim: int = 512, latent_dim: int = 2):
        super(VAE, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        
        # ---- エンコーダ ----
        # x → 隠れ表現 h → (μ, log σ²)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        # 平均 μ
        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)
        # 対数分散 log σ²
        self.fc_log_var = nn.Linear(hidden_dim // 2, latent_dim)
        
        # ---- デコーダ ----
        # z → 隠れ表現 → x̂
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),  # ピクセル値を [0, 1] に正規化
        )
    
    def encode(self, x: torch.Tensor):
        """エンコーダ: 入力 x から潜在分布のパラメータ (μ, log σ²) を計算
        
        Args:
            x: 入力テンソル [batch_size, input_dim]
        Returns:
            mu:      平均 μ [batch_size, latent_dim]
            log_var: 対数分散 log σ² [batch_size, latent_dim]
        """
        h = self.encoder(x)
        mu = self.fc_mu(h)
        log_var = self.fc_log_var(h)
        return mu, log_var
    
    def reparameterize(self, mu: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
        """再パラメータ化トリック: z = μ + ε·σ, ε ~ N(0, I)
        
        学習時はサンプリング、推論時は μ をそのまま返すこともできるが、
        ここでは学習・推論ともに同じ処理を使用。
        
        Args:
            mu:      平均 [batch_size, latent_dim]
            log_var: 対数分散 [batch_size, latent_dim]
        Returns:
            z: サンプリングされた潜在変数 [batch_size, latent_dim]
        """
        if self.training:
            # σ = exp(log σ² / 2) = exp(log σ)
            std = torch.exp(0.5 * log_var)
            # ε ~ N(0, I)
            eps = torch.randn_like(std)
            # z = μ + ε·σ
            return mu + eps * std
        else:
            # 推論時は μ をそのまま使う（決定論的な推論）
            return mu
    
    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """デコーダ: 潜在変数 z から再構成画像を生成
        
        Args:
            z: 潜在変数 [batch_size, latent_dim]
        Returns:
            x_recon: 再構成画像 [batch_size, input_dim]
        """
        return self.decoder(z)
    
    def forward(self, x: torch.Tensor):
        """順伝播
        
        Args:
            x: 入力テンソル [batch_size, input_dim]
        Returns:
            x_recon: 再構成画像 [batch_size, input_dim]
            mu:      潜在分布の平均 [batch_size, latent_dim]
            log_var: 潜在分布の対数分散 [batch_size, latent_dim]
        """
        # エンコード
        mu, log_var = self.encode(x)
        # 再パラメータ化
        z = self.reparameterize(mu, log_var)
        # デコード
        x_recon = self.decode(z)
        return x_recon, mu, log_var


def vae_loss(
    x_recon: torch.Tensor,
    x: torch.Tensor,
    mu: torch.Tensor,
    log_var: torch.Tensor,
    beta: float = 1.0
) -> tuple:
    """VAE の損失関数（負の ELBO）
    
    ELBO = E[log p(x|z)] - KL[q(z|x) || p(z)]
    損失 = -ELBO = 再構成誤差 + β × KL 損失
    
    Args:
        x_recon: 再構成画像 [batch_size, input_dim]
        x:       元の画像   [batch_size, input_dim]
        mu:      潜在分布の平均 [batch_size, latent_dim]
        log_var: 潜在分布の対数分散 [batch_size, latent_dim]
        beta:    KL 損失の重み（デフォルト: 1.0）
    Returns:
        total_loss: 合計損失
        recon_loss: 再構成誤差
        kl_loss:    KL ダイバージェンス
    """
    # 再構成誤差: Binary Cross Entropy
    # reduction='sum': バッチ内の全ピクセルの BCE の和
    recon_loss = F.binary_cross_entropy(x_recon, x, reduction='sum')
    
    # KL ダイバージェンス: KL[N(μ, σ²) || N(0, 1)]
    # = -1/2 * Σ(1 + log σ² - μ² - σ²)
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    
    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss


# モデルの概要を確認
LATENT_DIM = 2
model = VAE(input_dim=784, hidden_dim=512, latent_dim=LATENT_DIM).to(device)
print('VAE モデル構造:')
print(model)
print(f'\n総パラメータ数: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# データの読み込み（Fashion-MNIST）

# Fashion-MNIST のクラス名
CLASS_NAMES = [
    'T-shirt/top',  # 0
    'Trouser',      # 1
    'Pullover',     # 2
    'Dress',        # 3
    'Coat',         # 4
    'Sandal',       # 5
    'Shirt',        # 6
    'Sneaker',      # 7
    'Bag',          # 8
    'Ankle boot',   # 9
]

# データ変換: PIL → Tensor → [0, 1] に正規化 → 784 次元に平坦化
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))  # 28x28 → 784
])

# データセット読み込み
train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)
test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# DataLoader
BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'学習データ数: {len(train_dataset):,}')
print(f'テストデータ数: {len(test_dataset):,}')
print(f'バッチサイズ: {BATCH_SIZE}')
print(f'学習バッチ数: {len(train_loader)}')

# サンプル画像の確認
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    # 各クラスから1枚ずつ取得
    idx = next(j for j, (_, label) in enumerate(train_dataset) if label == i)
    img, label = train_dataset[idx]
    axes[0, i].imshow(img.view(28, 28), cmap='gray')
    axes[0, i].set_title(CLASS_NAMES[label], fontsize=7)
    axes[0, i].axis('off')
    axes[1, i].axis('off')

fig.suptitle('Fashion-MNIST サンプル画像（各クラス1枚）', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 学習ループ

# ハイパーパラメータ
EPOCHS = 10
LEARNING_RATE = 1e-3
BETA = 1.0  # KL 損失の重み（β-VAE のパラメータ）

# オプティマイザ
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 損失の記録
train_losses = []
train_recon_losses = []
train_kl_losses = []
test_losses = []


def train_epoch(model, loader, optimizer, device, beta):
    """1エポックの学習"""
    model.train()
    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0
    n_samples = 0
    
    for batch_idx, (x, _) in enumerate(loader):
        x = x.to(device)
        batch_size = x.size(0)
        
        # 順伝播
        x_recon, mu, log_var = model(x)
        
        # 損失計算
        loss, recon_loss, kl_loss = vae_loss(x_recon, x, mu, log_var, beta=beta)
        
        # 逆伝播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss  += loss.item()
        total_recon += recon_loss.item()
        total_kl    += kl_loss.item()
        n_samples   += batch_size
    
    # サンプルあたりの平均損失
    return total_loss / n_samples, total_recon / n_samples, total_kl / n_samples


def eval_epoch(model, loader, device, beta):
    """1エポックの評価"""
    model.eval()
    total_loss = 0.0
    n_samples = 0
    
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            x_recon, mu, log_var = model(x)
            loss, _, _ = vae_loss(x_recon, x, mu, log_var, beta=beta)
            total_loss += loss.item()
            n_samples  += x.size(0)
    
    return total_loss / n_samples


print('学習開始...')
print(f'エポック数: {EPOCHS}, 学習率: {LEARNING_RATE}, β: {BETA}')
print('-' * 70)
print(f'{"Epoch":>6} | {"Train Loss":>12} | {"Recon Loss":>12} | {"KL Loss":>10} | {"Test Loss":>10}')
print('-' * 70)

for epoch in range(1, EPOCHS + 1):
    # 学習
    train_loss, recon_loss, kl_loss = train_epoch(model, train_loader, optimizer, device, BETA)
    # 評価
    test_loss = eval_epoch(model, test_loader, device, BETA)
    
    train_losses.append(train_loss)
    train_recon_losses.append(recon_loss)
    train_kl_losses.append(kl_loss)
    test_losses.append(test_loss)
    
    print(f'{epoch:>6} | {train_loss:>12.4f} | {recon_loss:>12.4f} | {kl_loss:>10.4f} | {test_loss:>10.4f}')

print('-' * 70)
print('学習完了！')

# 損失曲線のプロット
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, train_losses, 'b-o', label='Train')
axes[0].plot(epochs_range, test_losses, 'r-o', label='Test')
axes[0].set_title('合計損失（ELBO）')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (per sample)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, train_recon_losses, 'g-o')
axes[1].set_title('再構成損失（BCE）')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (per sample)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, train_kl_losses, 'm-o')
axes[2].set_title('KL ダイバージェンス')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('KL Divergence (per sample)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('VAE 学習曲線（Fashion-MNIST, 10 epochs）', fontsize=13)
plt.tight_layout()
plt.show()

## 2. 再構成結果の可視化

テストデータから10枚サンプリングし、元画像（上段）と VAE で再構成した画像（下段）を比較します。

VAE は入力を一度 2 次元の潜在空間に圧縮してから復元しているため、若干ぼやけた再構成になりますが、全体的な形状はよく保持されています。

In [ ]:
# 再構成結果の可視化（元画像 vs 再構成画像）

model.eval()

# テストデータから10サンプルを取得
n_samples = 10
test_batch, test_labels = next(iter(test_loader))
test_batch = test_batch[:n_samples].to(device)
test_labels = test_labels[:n_samples]

with torch.no_grad():
    x_recon, mu, log_var = model(test_batch)

# CPU に戻して numpy 変換
originals   = test_batch.cpu().numpy().reshape(-1, 28, 28)
recons      = x_recon.cpu().numpy().reshape(-1, 28, 28)
labels      = test_labels.numpy()
latent_vecs = mu.cpu().numpy()

# 可視化
fig, axes = plt.subplots(3, n_samples, figsize=(18, 6))

for i in range(n_samples):
    # 元画像
    axes[0, i].imshow(originals[i], cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(CLASS_NAMES[labels[i]], fontsize=7, pad=2)
    axes[0, i].axis('off')
    
    # 再構成画像
    axes[1, i].imshow(recons[i], cmap='gray', vmin=0, vmax=1)
    axes[1, i].axis('off')
    
    # 差分（絶対値）
    diff = np.abs(originals[i] - recons[i])
    axes[2, i].imshow(diff, cmap='hot', vmin=0, vmax=1)
    axes[2, i].axis('off')

# 行ラベル
axes[0, 0].set_ylabel('元画像', fontsize=10, rotation=90, labelpad=5)
fig.text(0.01, 0.68, '元画像', va='center', fontsize=10)
fig.text(0.01, 0.38, '再構成', va='center', fontsize=10)
fig.text(0.01, 0.12, '差分', va='center', fontsize=10)

fig.suptitle('VAE 再構成結果（上：元画像 / 中：再構成 / 下：差分）', fontsize=12)
plt.tight_layout(rect=[0.03, 0, 1, 0.95])
plt.show()

print('潜在変数 z（μ）の値:')
for i in range(n_samples):
    print(f'  [{CLASS_NAMES[labels[i]]:<14}] z = ({latent_vecs[i, 0]:+.3f}, {latent_vecs[i, 1]:+.3f})')

## 3. 潜在空間の 2D 可視化

VAE の重要な特性のひとつは、**構造化された潜在空間**を学習することです。

latent_dim=2 に設定したため、潜在変数 z を直接 2D 散布図にプロットできます。

- 同じクラスの画像は潜在空間上の近い場所に集まる
- クラス間の境界は滑らか（連続的な遷移）
- 全体が概ね N(0, I) に従う分布になっている（KL 正則化の効果）

In [ ]:
# 潜在空間の 2D 可視化（テストデータ全体）

model.eval()

all_mu = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        mu, _ = model.encode(x)
        all_mu.append(mu.cpu().numpy())
        all_labels.append(y.numpy())

all_mu = np.concatenate(all_mu, axis=0)       # (10000, 2)
all_labels = np.concatenate(all_labels, axis=0)  # (10000,)

print(f'エンコードされた潜在変数の形状: {all_mu.shape}')
print(f'μ の統計: mean={all_mu.mean():.3f}, std={all_mu.std():.3f}')
print(f'μ の範囲: [{all_mu.min():.3f}, {all_mu.max():.3f}]')

# カラーマップ
cmap = plt.cm.get_cmap('tab10', 10)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- 左: 全クラスの散布図 ---
ax = axes[0]
for class_id in range(10):
    mask = all_labels == class_id
    ax.scatter(
        all_mu[mask, 0],
        all_mu[mask, 1],
        c=[cmap(class_id)],
        label=CLASS_NAMES[class_id],
        alpha=0.4,
        s=5,
        edgecolors='none'
    )
ax.set_title('潜在空間 z（μ）の分布\n（テストデータ 10,000 サンプル）', fontsize=11)
ax.set_xlabel('z₁')
ax.set_ylabel('z₂')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8, markerscale=3)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# 事前分布 N(0, I) の等高線を重ね書き
theta = np.linspace(0, 2 * np.pi, 100)
for r in [1, 2, 3]:
    ax.plot(r * np.cos(theta), r * np.sin(theta), 'k--', alpha=0.2, linewidth=1)
    ax.text(r * 0.707, r * 0.707, f'σ={r}', fontsize=7, color='gray')

# --- 右: クラス中心のみをプロット ---
ax2 = axes[1]
for class_id in range(10):
    mask = all_labels == class_id
    center = all_mu[mask].mean(axis=0)
    ax2.scatter(
        all_mu[mask, 0],
        all_mu[mask, 1],
        c=[cmap(class_id)],
        alpha=0.2,
        s=3,
        edgecolors='none'
    )
    ax2.annotate(
        CLASS_NAMES[class_id],
        center,
        fontsize=8,
        fontweight='bold',
        ha='center',
        bbox=dict(boxstyle='round,pad=0.2', facecolor=cmap(class_id), alpha=0.7)
    )

ax2.set_title('潜在空間 z（クラスラベル付き）', fontsize=11)
ax2.set_xlabel('z₁')
ax2.set_ylabel('z₂')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.suptitle('VAE 潜在空間の可視化（Fashion-MNIST）', fontsize=13)
plt.tight_layout()
plt.show()

## 4. 潜在空間からのサンプリング（マニフォールド可視化）

VAE の最大の特徴は**生成モデル**であることです。

潜在空間 z を 2D グリッド状にサンプリングし、各点からデコーダで画像を生成します。
これにより、潜在空間上でどのような画像が配置されているかを確認できます。

このような可視化を **マニフォールド可視化**（または **latent space traversal**）と呼びます。

VAE が学習した潜在空間では：
- 連続的に z を変化させると、生成画像も滑らかに変化する
- 空白地帯がなく、どの点からも意味のある画像が生成できる

In [ ]:
# 潜在空間からのサンプリング（マニフォールド可視化）

model.eval()

# グリッドの設定
n_grid = 20  # 20×20 = 400 サンプル

# 潜在空間の範囲（正規分布の分位数を使って意味のある範囲を選択）
from scipy.stats import norm
z_range = np.linspace(0.05, 0.95, n_grid)  # 分位数
z_range = norm.ppf(z_range)                # 正規分布の逆関数（分位点関数）

# 全グリッド点での画像を生成
canvas = np.zeros((28 * n_grid, 28 * n_grid))

with torch.no_grad():
    for i, z2 in enumerate(z_range[::-1]):   # y 軸（上が正）
        for j, z1 in enumerate(z_range):      # x 軸
            z = torch.tensor([[z1, z2]], dtype=torch.float32).to(device)
            x_gen = model.decode(z)
            img = x_gen.cpu().numpy().reshape(28, 28)
            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = img

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- 左: マニフォールド ---
im = axes[0].imshow(
    canvas,
    cmap='gray',
    extent=[z_range.min(), z_range.max(), z_range.min(), z_range.max()],
    aspect='equal',
    interpolation='nearest'
)
axes[0].set_title(f'VAE 潜在空間マニフォールド\n（{n_grid}×{n_grid} グリッド）', fontsize=11)
axes[0].set_xlabel('z₁')
axes[0].set_ylabel('z₂')

# 事前分布の等高線
theta = np.linspace(0, 2 * np.pi, 100)
for r in [1, 2, 3]:
    axes[0].plot(r * np.cos(theta), r * np.sin(theta), 'r--', alpha=0.4, linewidth=1)

# --- 右: テストデータの潜在空間分布を重ねる ---
axes[1].imshow(
    canvas,
    cmap='gray',
    extent=[z_range.min(), z_range.max(), z_range.min(), z_range.max()],
    aspect='equal',
    interpolation='nearest',
    alpha=0.7
)
# テストデータのエンコード結果を重ねる（各クラス 50 サンプル）
for class_id in range(10):
    mask = all_labels == class_id
    idx = np.where(mask)[0][:50]  # 各クラス 50 サンプル
    axes[1].scatter(
        all_mu[idx, 0],
        all_mu[idx, 1],
        c=[cmap(class_id)],
        label=CLASS_NAMES[class_id],
        s=15,
        alpha=0.8,
        edgecolors='white',
        linewidth=0.3
    )
axes[1].set_title('マニフォールド + テストデータの z 分布', fontsize=11)
axes[1].set_xlabel('z₁')
axes[1].set_ylabel('z₂')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7, markerscale=2)

plt.suptitle('VAE 潜在空間の可視化とマニフォールド', fontsize=13)
plt.tight_layout()
plt.show()

print(f'マニフォールドの z 範囲: [{z_range.min():.2f}, {z_range.max():.2f}]')
print(f'生成画像の総数: {n_grid * n_grid}')

In [ ]:
# 追加: ランダムサンプリングによる新規画像生成

model.eval()

# 事前分布 p(z) = N(0, I) からサンプリング
n_gen = 64
z_samples = torch.randn(n_gen, LATENT_DIM).to(device)

with torch.no_grad():
    generated = model.decode(z_samples)

generated = generated.cpu().numpy().reshape(-1, 28, 28)

# 可視化
fig, axes = plt.subplots(8, 8, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i], cmap='gray', vmin=0, vmax=1)
    ax.axis('off')

fig.suptitle(f'VAE でランダム生成した画像\n（事前分布 N(0,I) から {n_gen} サンプル）', fontsize=12)
plt.tight_layout()
plt.show()


# 追加: 2つの画像の補間（interpolation）
print('\n--- 潜在空間上での2画像の補間 ---')

# テストデータから Sneaker と T-shirt を取得
img_a, img_b = None, None
for x, y in test_loader:
    for i in range(len(y)):
        if img_a is None and y[i].item() == 7:  # Sneaker
            img_a = x[i:i+1]
        if img_b is None and y[i].item() == 0:  # T-shirt
            img_b = x[i:i+1]
        if img_a is not None and img_b is not None:
            break
    if img_a is not None and img_b is not None:
        break

img_a = img_a.to(device)
img_b = img_b.to(device)

# 各画像の潜在表現を取得
with torch.no_grad():
    mu_a, _ = model.encode(img_a)  # Sneaker
    mu_b, _ = model.encode(img_b)  # T-shirt

# 線形補間
n_interp = 10
alphas = np.linspace(0, 1, n_interp)
interpolated_images = []

with torch.no_grad():
    for alpha in alphas:
        z_interp = (1 - alpha) * mu_a + alpha * mu_b
        img_gen = model.decode(z_interp)
        interpolated_images.append(img_gen.cpu().numpy().reshape(28, 28))

# 可視化
fig, axes = plt.subplots(1, n_interp + 2, figsize=(16, 2))

axes[0].imshow(img_a.cpu().numpy().reshape(28, 28), cmap='gray')
axes[0].set_title('Sneaker\n(α=0)', fontsize=8)
axes[0].axis('off')

for i, (img, alpha) in enumerate(zip(interpolated_images, alphas)):
    axes[i+1].imshow(img, cmap='gray', vmin=0, vmax=1)
    axes[i+1].set_title(f'α={alpha:.1f}', fontsize=7)
    axes[i+1].axis('off')

axes[-1].imshow(img_b.cpu().numpy().reshape(28, 28), cmap='gray')
axes[-1].set_title('T-shirt\n(α=1)', fontsize=8)
axes[-1].axis('off')

fig.suptitle('潜在空間上での2画像の線形補間（Sneaker → T-shirt）', fontsize=11)
plt.tight_layout()
plt.show()

## 5. まとめと参考文献

### まとめ

このノートブックでは、VAE（変分オートエンコーダ）を Fashion-MNIST データセットで実装・学習しました。

#### 実装のポイント

1. **エンコーダ**: 入力 x から潜在分布のパラメータ (μ, log σ²) を出力
2. **再パラメータ化トリック**: z = μ + ε·σ (ε ~ N(0,I)) で微分可能なサンプリングを実現
3. **損失関数（負の ELBO）**:
   - 再構成損失（BCE）: 入力画像を正確に復元する
   - KL 損失: 潜在分布を N(0,I) に近づける正則化
4. **デコーダ**: 潜在変数 z から再構成画像を生成

#### 観察できた特性

- **構造化された潜在空間**: 同じクラスの画像は潜在空間上で近い場所に集まる
- **連続的な遷移**: 潜在空間上を移動すると、生成画像が滑らかに変化する
- **補間能力**: 2つの異なる画像を潜在空間上で補間すると、中間的な画像が生成できる

#### VAE の限界と発展

| 課題 | 解決アプローチ |
|---|---|
| 生成画像のぼやけ | VQVAE, β-VAE の改善 |
| 表現力の限界（ガウス分布仮定） | Normalizing Flow VAE |
| より高品質な生成 | GAN, Diffusion Models との組み合わせ |
| 潜在空間の解釈性 | β-VAE (β > 1) で改善 |

---

### 次のステップ

- **β-VAE**: β > 1 にして解釈性と生成品質のトレードオフを試す
- **Conv VAE**: 線形層の代わりに畳み込み層を使って画像の空間的構造を活用
- **Conditional VAE (CVAE)**: クラスラベルを条件として制御された生成
- **VQ-VAE**: 離散的な潜在空間を持つ VAE の発展形

---

### 参考文献

1. **Kingma & Welling (2013)**. *Auto-Encoding Variational Bayes*. https://arxiv.org/abs/1312.6114
   - VAE の原論文。ELBO の導出と再パラメータ化トリックの詳細。

2. **Doersch (2016)**. *Tutorial on Variational Autoencoders*. https://arxiv.org/abs/1606.05908
   - 数学的な背景を丁寧に解説したチュートリアル。

3. **Higgins et al. (2017)**. *β-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework*. ICLR 2017.
   - β-VAE の提案論文。潜在空間の解釈性改善。

4. **van den Oord et al. (2017)**. *Neural Discrete Representation Learning (VQ-VAE)*. https://arxiv.org/abs/1711.00937
   - 離散的な潜在空間を持つ VQ-VAE の提案。

5. **PyTorch 公式チュートリアル**: https://pytorch.org/tutorials/

---

*このノートブックは `mlpg` シリーズの一部です。*
*前: [126. Attention Bridge to Transformer](../sequence-models/126_attention_bridge_to_transformer_v1.ipynb)*